In [1]:
from utils_phoneme_reco import *


/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


# Wav2vec+CTC

In [2]:

MODEL_ID = "/vol/experiments3/imbenamor/TAPAS-FRAIS/models/wav2vec2-french-phonemizer"

model = AutoModelForCTC.from_pretrained(MODEL_ID)
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)

device = "cuda"
model = model.to(device)
model.eval()


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

# WavLM+CTC

In [2]:
import torch, librosa
from transformers import WavLMForCTC, Wav2Vec2FeatureExtractor, Wav2Vec2PhonemeCTCTokenizer

#CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme/checkpoint-268600"
CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme-large/checkpoint-476220"
ROOT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme-large"          # where vocab.json / tokenizer were saved

device = "cuda"
model = WavLMForCTC.from_pretrained(CKPT).to(device).eval()
feat  = Wav2Vec2FeatureExtractor.from_pretrained(CKPT)   # also present in CKPT
tok   = Wav2Vec2PhonemeCTCTokenizer.from_pretrained(ROOT)
tok.unk_token = "[UNK]"
tok.pad_token = "[PAD]"
model = model.to(device)
model.eval()

WavLMForCTC(
  (wavlm): WavLMModel(
    (feature_extractor): WavLMFeatureEncoder(
      (conv_layers): ModuleList(
        (0): WavLMLayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x WavLMLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x WavLMLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): WavLMFeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
 

# Whisper+CTC

In [2]:
#whisper
from transformers import WhisperFeatureExtractor, Wav2Vec2PhonemeCTCTokenizer

CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/whisper-fr-phoneme/checkpoint-193924"
ROOT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/whisper-fr-phoneme"

device = "cuda"

model = WhisperEncoderForCTC.from_pretrained(CKPT).to(device).eval()
feat  = WhisperFeatureExtractor.from_pretrained(CKPT)
tok   = Wav2Vec2PhonemeCTCTokenizer.from_pretrained(ROOT)
tok.unk_token = "[UNK]"
tok.pad_token = "[PAD]"

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'WhisperTokenizer'. 
The class this function is called from is 'Wav2Vec2PhonemeCTCTokenizer'.


In [3]:
import os
import json
from pathlib import Path
import pandas as pd
import pickle

audio_dir = "/vol/corpora/TAPAS_FRAIS/Data_Partagees_Mons/Data_Mons/Description"
textgrid_dir = Path("/vol/corpora/TAPAS_FRAIS/Data_Partagees_Mons/Mons TextGrid verifie + 50 ans" )
ref_files = textgrid_dir.glob("*") 

ref_dict = {f.stem:f for f in ref_files}

model_name = "whisper"

ref_inventory = set()
hyp_inventory = set()
alignment_store = {}

for filepath in textgrid_dir.glob("*"):
    f = filepath.stem + ".wav"
    audio_path = os.path.join(audio_dir, f)

    if os.path.exists(audio_path):


        textgrid_path = os.path.join(
            textgrid_dir,
            f.replace(".wav", ".TextGrid")
        )
        if model_name =="whisper":
            pred_phonemes, pred_alignments = get_phoneme_alignments_whisper_ctcfa(model, feat,tok, audio_path)
        elif model_name == "wavlm":
            pred_phonemes, pred_alignments = get_phoneme_alignments_wavlm_ctcfa(model, feat,tok, audio_path)
        else:
            pred_phonemes, pred_alignments = get_phoneme_alignments_w2v_ctcfa(model, processor, audio_path)

        ref_alignments = get_reference_alignments(textgrid_path, t="MAU")

        clean_hyp = clean_alignment_dict(pred_alignments, is_hyp=True)
        clean_ref = clean_alignment_dict(
            ref_alignments,
            flag="",
            is_hyp=False
        )

        ref_seq = extract_phoneme_sequence(clean_ref)
        hyp_seq = extract_phoneme_sequence(clean_hyp)

        # collect inventories
        for ph in ref_seq:
            ref_inventory.add(ph)

        for ph in hyp_seq:
            hyp_inventory.add(ph)

        entry = {
            "file": f,
            "ref_intervals": clean_ref,
            "hyp_intervals": clean_hyp,
            "ref_seq": ref_seq,
            "hyp_seq": hyp_seq,
        }

        alignment_store[f] = entry



Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`
/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/vol/experiments3/imbenamor/TAPAS-FRAIS/src/utils/utils_phoneme_reco.py:120: UserWarning: torchaudio.functional._alignment.forced_align has been deprecated. This deprecation is part of a large re

In [4]:
# ==========================
# Inventory comparison
# ==========================

print(f"Reference inventory size: {len(ref_inventory)}")
print(f"Hypothesis inventory size: {len(hyp_inventory)}")

only_in_ref = sorted(ref_inventory - hyp_inventory)
only_in_hyp = sorted(hyp_inventory - ref_inventory)

if not only_in_ref and not only_in_hyp:
    print("\n✓ REF and HYP inventories are identical.")
else:
    print("\n✗ Inventories differ.")

    if only_in_ref:
        print("\nPhonemes present only in REF:")
        print(only_in_ref)

    if only_in_hyp:
        print("\nPhonemes present only in HYP:")
        print(only_in_hyp)

# Optional: print full inventories
print("\nREF inventory:")
print(sorted(ref_inventory))

print("\nHYP inventory:")
print(sorted(hyp_inventory))

Reference inventory size: 34
Hypothesis inventory size: 33

✗ Inventories differ.

Phonemes present only in REF:
['ɥ']

REF inventory:
['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'œ', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɥ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']

HYP inventory:
['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'œ', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']


In [5]:
from metrics import *
import pickle
from jiwer import process_words
# (optional) save back so you don't redo it
with open(f"ctc_results/alignment_{model_name}_mon.pkl", "wb") as f:
    pickle.dump(alignment_store, f)


In [6]:
from metrics_alignment import *

with open(f"ctc_results/alignment_{model_name}_mon.pkl", "rb") as f:
    alignment_store = pickle.load(f)
compute_metrics(alignment_store, f"ctc_results/metrics++_{model_name}_mon.csv", per_phoneme_csv=None)


,group,style,K,AAS (ms),Median_start (ms),Mean_start (ms),Median_end (ms),Mean_end (ms),P90 (ms),%>50ms,...,Median_dur (ms),%dur>50ms,N_ref,N_hyp,PER (%),Deletions,Insertions,F1@0ms (%),F1@20ms (%),F1@50ms (%)
0,MO_H_ER48_2017_01_28_ModuleDescription_descrip...,ALL,215,80.072291,39.6875,47.122360,45.7325,113.022221,95.9375,39.534884,...,30.0,29.302326,278,259,24.100719,23,4,0.0,32.774674,68.901304
1,MO_F_ER19_2016_04_10_ModuleDescription_descrip...,ALL,186,70.054704,43.4375,45.244247,47.1875,94.865161,93.4375,43.817204,...,30.0,31.182796,207,206,13.043478,7,6,0.0,28.087167,69.733656
2,MO_H_SD36_2016_06_23_ModuleDescription_descrip...,ALL,339,88.208046,52.8125,68.808960,56.5625,107.607131,128.4375,54.277286,...,30.0,34.513274,402,389,18.656716,25,12,0.0,25.537295,61.694058
3,MO_H_SD11_2016_04_25_ModuleDescription_descrip...,ALL,531,81.063413,48.4375,54.379308,49.0625,107.747519,117.1265,46.704331,...,30.0,30.508475,595,598,14.789916,21,24,0.0,31.349539,72.254820
4,MO_H_AV18_2016_04_18_ModuleDescription_descrip...,ALL,176,76.736136,47.8125,52.296136,51.5625,101.176136,100.3125,50.284091,...,30.0,25.568182,208,205,19.230769,11,8,0.0,22.760291,65.375303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,MO_H_SD30_2016_05_21_ModuleDescription_descrip...,ALL,339,72.682611,41.5625,59.901630,41.5625,85.463591,88.4375,39.380531,...,20.0,19.174041,398,400,21.105528,23,25,0.0,37.092732,76.441103
66,MO_F_AV21_2016_04_18_ModuleDescription_descrip...,ALL,299,85.245109,47.8125,66.443637,47.8125,104.046580,109.5685,44.481605,...,30.0,27.759197,365,371,24.657534,18,24,0.0,27.173913,68.478261
67,MO_H_ER49_2017_01_28_ModuleDescription_descrip...,ALL,179,68.380922,34.0625,53.378226,36.5625,83.383617,80.3150,27.653631,...,30.0,22.905028,229,258,39.737991,12,41,0.0,32.854209,71.457906
68,STYLE_ALL,ALL,25300,74.494518,44.0625,55.939037,46.5625,93.050000,104.6875,43.897233,...,30.0,28.296443,30592,29958,21.701752,1981,1347,0.0,30.483898,69.859620


In [7]:
dict_to_csv(alignment_store, f'ctc_results/pred_{model_name}_mon.csv')

# Prepare for MFA

In [8]:
df1=pd.read_csv(f"/vol/experiments3/imbenamor/TAPAS-FRAIS/src/utils/ctc_results/pred_{model_name}_mon.csv")
df1["speaker_id"] = ["_".join(i.split("_")[:3]) for i in df1["filename"]]
df1

,filename,predicted_phonemes,speaker_id
0,MO_H_ER48_2017_01_28_ModuleDescription_descrip...,a l ɔ ʁ s ɛ t p j ɛ s o a t e a t ʁ a v ɛ k ə ...,MO_H_ER48
1,MO_F_ER19_2016_04_10_ModuleDescription_descrip...,l ə l e s a k a p o k u ʒ s ə t ʁ u v d ɑ̃ l ə...,MO_F_ER19
2,MO_H_SD36_2016_06_23_ModuleDescription_descrip...,a t e a t ʁ a v ɛ k a m a ʒ i s j ɛ̃ k i p ɔ ʁ...,MO_H_SD36
3,MO_H_SD11_2016_04_25_ModuleDescription_descrip...,a s ə l a p ʁ e z ɑ̃ t t y n s ɛ n d ə t e a s...,MO_H_SD11
4,MO_H_AV18_2016_04_18_ModuleDescription_descrip...,b ɔ̃ s ɛ t ɛ̃ s i ʒ ə v w a z a m a ʒ i s j ɛ̃...,MO_H_AV18
...,...,...,...
63,MO_H_ER43_2017_05_23_ModuleDescription_descrip...,o n i v a m k ɔ m ɑ̃ t e d ɔ̃ k p w ɛ̃ a l y i...,MO_H_ER43
64,MO_F_AV17_2016_04_18_ModuleDescription_descrip...,a l ɔ ʁ ʁ ʒ ɛ a ʁ i d o d ə t e a t ʁ a l ɔ ʁ ...,MO_F_AV17
65,MO_H_SD30_2016_05_21_ModuleDescription_descrip...,d ɔ̃ k ɑ̃ f ɛ t j l a m a ʒ i s j ɛ̃ k i f ɛ s...,MO_H_SD30
66,MO_F_AV21_2016_04_18_ModuleDescription_descrip...,ʒ ə v w a y n s ɛ n d ə t e a t ʁ a v ɛ k l ə ...,MO_F_AV21


In [9]:
df1["predicted_phonemes"] = df1["predicted_phonemes"].apply(normalize_phones)
ref_inventory = set()

for seq in df1["predicted_phonemes"].dropna():
    tokens = seq.split()
    ref_inventory.update(tokens)

print(sorted(ref_inventory))
print("Number of unique phonemes:", len(ref_inventory))

['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'œ', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']
Number of unique phonemes: 33


In [10]:
#Generation des fichiers pour mFA
import os
import torch
import torchaudio
import shutil

corpus_dir = f"/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/mfa_mon_{model_name}_ctc"
if os.path.exists(corpus_dir):
    shutil.rmtree(corpus_dir)  
os.makedirs(corpus_dir)
target_sr = 16000
wav_path = "/vol/corpora/TAPAS_FRAIS/Data_Partagees_Mons/Data_Mons/Description"
for idx, row in df1.iterrows():
    audio_path = os.path.join(wav_path, row["filename"])
    tokens = row["predicted_phonemes"]
    utt_id = os.path.splitext(os.path.basename(audio_path))[0]
    speaker_id = str(row["speaker_id"])  

    # Create speaker folder
    speaker_dir = os.path.join(corpus_dir, speaker_id)
    os.makedirs(speaker_dir, exist_ok=True)
    # Load audio
    waveform, sr = torchaudio.load(audio_path)
    # Resample if needed
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        waveform = resampler(waveform)

    # Save wav inside speaker folder
    torchaudio.save(
        os.path.join(speaker_dir, f"{utt_id}.wav"),
        waveform,
        target_sr
    )

    # Save lab inside speaker folder
    with open(os.path.join(speaker_dir, f"{utt_id}.lab"), "w", encoding="utf-8") as f:
        f.write(tokens.strip())

/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch

In [11]:
with open(f"/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/output_ph_reco/phoneme_mon_{model_name}_ctc.txt", "w", encoding="utf-8") as f:
    for ph in ref_inventory:
        f.write(f"{ph} {ph}\n")

# MFA ALIGN: GO to cmd 

# MFA alignment

In [17]:
#rhapsodie

import os
import json
from pathlib import Path
import pandas as pd
import pickle
from praatio import textgrid
import os
import json
from pathlib import Path
import pandas as pd
import pickle

tg_path = Path(f"/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/align_{model_name}_mon")
ref_tg_path = Path("/vol/corpora/TAPAS_FRAIS/Data_Partagees_Mons/Mons TextGrid verifie + 50 ans")
audio_dir = "/vol/corpora/TAPAS_FRAIS/Data_Partagees_Mons/Data_Mons/Description"
ref_files = textgrid_dir.glob("*")
full_paths = list(tg_path.rglob("*.TextGrid"))

alignment_store = {}
for hyp_path in full_paths:
    if ".ipynb_checkpoints" not in str(hyp_path):
        stem=hyp_path.stem
        #stem = hyp_path.stem.split("-")[1]+"-"+hyp_path.stem.split("-")[0] #txtgrid from mfa
        if stem not in ref_dict.keys():
            continue
    
        ref_path = ref_dict[stem]
        
        #f = hyp_path.stem.split("-")[1]+"-"+hyp_path.stem.split("-")[0] + ".wav"
        #f = stem.split("-")[1]+"-"+stem.split("-")[0] + ".wav"
        audio_path = os.path.join(audio_dir, stem+".wav")
        if os.path.exists(audio_path):
            phones_hyp, hyp_intervals = extract_phones_from_textgrid(hyp_path, t="phones")
            phones_ref, ref_intervals = extract_phones_from_textgrid(ref_path, t="MAU")
            pred_alignments=[]
            for i,j,k in hyp_intervals:
                pred_alignments.append({"phoneme":k,"start":i,"end":j})
            ref_alignments=[]
            for i,j,k in ref_intervals:
                ref_alignments.append({"phoneme":k,"start":i,"end":j})
            clean_hyp = clean_alignment_dict(pred_alignments, is_hyp=True)
            clean_ref = clean_alignment_dict(ref_alignments, is_hyp=False)
            

            entry = {
                "file":          stem,
                "ref_intervals": clean_ref,
                "hyp_intervals": clean_hyp,
                "ref_seq":       extract_phoneme_sequence(clean_ref),
                "hyp_seq":       extract_phoneme_sequence(clean_hyp),
            }
            alignment_store[stem] = entry 

In [18]:
len(alignment_store)

68

In [19]:
from metrics import *
import pickle
from jiwer import process_words
# (optional) save back so you don't redo it
with open(f"mfa_results/alignment_{model_name}_mfa_mon.pkl", "wb") as f:
    pickle.dump(alignment_store, f)


In [20]:
from metrics_alignment import *

with open(f"mfa_results/alignment_{model_name}_mfa_mon.pkl", "rb") as f:
    alignment_store = pickle.load(f)
compute_metrics(alignment_store, f"mfa_results/metrics++_{model_name}_mfa_mon.csv", per_phoneme_csv=None)


,group,style,K,AAS (ms),Median_start (ms),Mean_start (ms),Median_end (ms),Mean_end (ms),P90 (ms),%>50ms,...,Median_dur (ms),%dur>50ms,N_ref,N_hyp,PER (%),Deletions,Insertions,F1@0ms (%),F1@20ms (%),F1@50ms (%)
0,MO_H_ER24_2016_04_15_ModuleDescription_descrip...,ALL,299,81.173130,20.000,81.114591,20.0000,81.231668,101.530211,22.909699,...,30.000,33.110368,434,337,32.718894,104,7,4.150454,46.952010,68.741894
1,MO_F_ER18_2016_04_08_ModuleDescription_descrip...,ALL,721,35.424386,10.002,33.146636,10.0040,37.702136,80.002900,16.504854,...,29.994,28.571429,859,782,18.859139,101,24,4.753199,61.182206,81.170018
2,MO_H_ER41_2017_05_29_ModuleDescription_descrip...,ALL,223,52.161496,19.999,48.740917,20.0000,55.582075,89.998500,22.197309,...,30.002,37.219731,290,251,26.206897,48,9,5.914972,55.822551,76.155268
3,MO_H_ER41_2017_01_24_ModuleDescription_descrip...,ALL,361,58.303158,19.998,54.252612,20.0000,62.353704,100.000000,25.207756,...,30.000,34.626039,438,398,20.091324,51,11,7.655502,59.330144,77.511962
4,MO_H_SD13_2016_08_13_ModuleDescription_descrip...,ALL,951,91.103110,10.005,94.126936,10.0050,88.079284,120.000200,18.296530,...,20.005,23.554154,1138,1030,18.980668,137,29,3.136531,60.885609,79.612546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,MO_H_ER48_2017_01_28_ModuleDescription_descrip...,ALL,200,101.846185,20.000,106.297677,19.9990,97.394693,151.260719,23.000000,...,30.000,34.000000,278,220,29.496403,62,4,6.024096,52.208835,71.084337
66,MO_F_AV01_2016_04_08_ModuleDescription_descrip...,ALL,516,45.277148,10.000,43.758541,10.0005,46.795755,50.000000,9.980620,...,20.000,17.248062,623,564,20.064205,77,18,8.256108,71.777591,85.930918
67,MO_F_AV27_2016_05_14_ModuleDescription_descrip...,ALL,359,69.019110,10.001,65.912259,10.0010,72.125962,100.002000,18.105850,...,20.000,19.777159,497,408,29.778672,99,10,10.386740,62.541436,77.790055
68,STYLE_ALL,ALL,23374,69.182140,10.001,68.207733,10.0020,70.156546,80.006000,16.225293,...,20.001,22.738941,30592,26057,26.526543,5432,897,8.169606,62.207629,78.963442
